In [1]:
# Cell 1: Imports and Directory Management
import sys
import os
import glob
import math
import pandas as pd
import pypowsybl as pp
import pypowsybl.network as pn
from unified_generator import generate_dynamic_files

NOTEBOOK_NAME = "OM_test_2"
OUTPUT_DIR = os.path.join(os.getcwd(), f"{NOTEBOOK_NAME}_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Environment ready. Outputs will be saved in: {OUTPUT_DIR}")

# Setup OMPython (Uncomment if installed)
# from OMPython import OMCSessionZMQ
# omc = OMCSessionZMQ()

Environment ready. Outputs will be saved in: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_2_outputs


In [2]:
# Cell 2: Create the BESS Network Topology
# Based on MyBESS_static.mo: InfiniteBus -- Line -- BESS(Gen)


def create_bess_network(s_base_mva=100.0, nom_v_kv=20.0):
    z_base = (nom_v_kv**2) / s_base_mva
    network = pp.network.create_empty()

    network.create_substations(id="Sub_Grid", name="Grid Side", country="FR", tso="TSO")
    network.create_substations(id="Sub_BESS", name="BESS Side", country="FR", tso="TSO")

    network.create_voltage_levels(
        id="Grid_VL", substation_id="Sub_Grid", topology_kind="BUS_BREAKER", nominal_v=nom_v_kv
    )
    network.create_voltage_levels(
        id="BESS_VL", substation_id="Sub_BESS", topology_kind="BUS_BREAKER", nominal_v=nom_v_kv
    )

    network.create_buses(id="Bus_Infinite", voltage_level_id="Grid_VL")
    network.create_buses(id="Bus_BESS", voltage_level_id="BESS_VL")

    network.create_lines(
        id="Line_Connect",
        voltage_level1_id="Grid_VL",
        bus1_id="Bus_Infinite",
        voltage_level2_id="BESS_VL",
        bus2_id="Bus_BESS",
        x=0.0000020661 * z_base,
        r=0.0,
    )

    network.create_generators(
        id="InfiniteBus",
        voltage_level_id="Grid_VL",
        bus_id="Bus_Infinite",
        target_v=1.0 * nom_v_kv,
        target_p=0.0,
        voltage_regulator_on=True,
        min_p=-9999.0,
        max_p=9999.0,
    )

    p_gen_pu = 0.03
    p_target_mw = p_gen_pu * s_base_mva

    network.create_generators(
        id="GenPV",
        voltage_level_id="BESS_VL",
        bus_id="Bus_BESS",
        target_p=p_target_mw,
        target_v=1.0 * nom_v_kv,
        voltage_regulator_on=True,
        min_p=-100.0,
        max_p=100.0,
        rated_s=s_base_mva,
    )

    return network


s_base = 100.0
nom_v = 110.0
network = create_bess_network(s_base_mva=s_base, nom_v_kv=nom_v)
print("BESS Network created in PyPowSyBl.")

BESS Network created in PyPowSyBl.


In [3]:
# Cell 3: Run Power Flow (Calculate Initial State)
import os

parameters = pp.loadflow.Parameters(
    distributed_slack=False,
    provider_parameters={"slackBusSelectionMode": "NAME", "slackBusesIds": "InfiniteBus"},
)

print("Running Loadflow...")
results = pp.loadflow.run_ac(network, parameters=parameters)

status_name = (
    results[0].status.name if hasattr(results[0].status, "name") else str(results[0].status)
)
if status_name == "CONVERGED":
    print(f"Power Flow Converged!")
else:
    print(f"Power Flow Failed: {status_name}")

# SAVE TO OUTPUT DIRECTORY
xiidm_path = os.path.join(OUTPUT_DIR, "bess_initialized.xiidm")
network.save(xiidm_path, format="XIIDM", parameters={"iidm.export.xml.version": "1.4"})
print(f"Saved initialized network to: {xiidm_path}")

Running Loadflow...
Power Flow Converged!
Saved initialized network to: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_2_outputs/bess_initialized.xiidm


In [4]:
# Cell 4: Generate Dynamic Configuration (Dynawo Files)
# Now using the file located in the output directory

xiidm_path = os.path.join(OUTPUT_DIR, "bess_initialized.xiidm")

if os.path.exists(xiidm_path):
    # The function generate_dynamic_files usually creates files in the same dir as the input file
    generate_dynamic_files(xiidm_path, simulator="dynawo")
    print(f"Dynamic files (.dyd and .par) generated successfully in: {OUTPUT_DIR}")
else:
    print(f"Error: Could not find '{xiidm_path}'. Please run Cell 3 again.")

--- Generating configuration for DYNAWO using /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_2_outputs/bess_initialized.xiidm ---
Successfully generated files for dynawo.
Dynamic files (.dyd and .par) generated successfully in: /home/guiu/Projects/dynawo-notebooks/Model_and_Params/OM_test_2_outputs


In [5]:
# Cell 5: Extract Data and Generate Initialization Commands


def get_initialization_values(network, gen_id):
    gens = network.get_generators(all_attributes=True)
    buses = network.get_buses(all_attributes=True)

    p_gen = gens.at[gen_id, "p"]
    q_gen = gens.at[gen_id, "q"]
    bus_id = gens.at[gen_id, "bus_id"]
    v_mag_kv = buses.at[bus_id, "v_mag"]
    v_angle_deg = buses.at[bus_id, "v_angle"]

    v_nom = network.get_voltage_levels().at[gens.at[gen_id, "voltage_level_id"], "nominal_v"]

    return {
        "P_gen_init": p_gen,
        "Q_gen_init": q_gen,
        "V_mag_pu": v_mag_kv / v_nom,
        "Angle_deg": v_angle_deg,
    }


init_vals = get_initialization_values(network, "GenPV")

print("--- Initialization Values for Modelica ---")
for key, val in init_vals.items():
    print(f"{key}: {val:.4f}")

print("\n--- Example OMJulia Command Generation ---")
om_commands = [
    f"parameter Real P0 = {init_vals['P_gen_init']};",
    f"parameter Real Q0 = {init_vals['Q_gen_init']};",
    f"parameter Real V0 = {init_vals['V_mag_pu']};",
    f"parameter Real A0 = {init_vals['Angle_deg']};",
    "simulate(YourModelName, stopTime=1.0)",
]

print("Commands ready to be sent to OMJulia:")
print("\n".join(om_commands))

--- Initialization Values for Modelica ---
P_gen_init: -3.0000
Q_gen_init: -0.0000
V_mag_pu: 1.0000
Angle_deg: 0.0000

--- Example OMJulia Command Generation ---
Commands ready to be sent to OMJulia:
parameter Real P0 = -3.0;
parameter Real Q0 = -0.0;
parameter Real V0 = 1.0;
parameter Real A0 = 0.0;
simulate(YourModelName, stopTime=1.0)
